In [3]:

import pandas as pd
import numpy as np
import os
import json
import pickle
import time
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Vector store
import faiss

# Embeddings (same as Task 2)
from sentence_transformers import SentenceTransformer

# For LLM
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import torch

# For evaluation
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

print("Loded successfully")

Loded successfully


In [4]:

print("\n Loading embedding model...")
try:
    # Try to load sentence transformer
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
    print(" Loaded all-MiniLM-L6-v2")
except:
    # Fallback to local model if available
    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.decomposition import TruncatedSVD
        from sklearn.preprocessing import normalize
        
        class LocalEmbeddingModel:
            def __init__(self, dimension=384):
                self.dimension = dimension
                self.vectorizer = None
                self.svd = None
                self.is_fitted = False
                
            def encode(self, texts, normalize_embeddings=True):
                if isinstance(texts, str):
                    texts = [texts]
                
                if not self.is_fitted:
                    # Fit on the fly
                    self.vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
                    self.vectorizer.fit(texts)
                    tfidf = self.vectorizer.transform(texts)
                    n_components = min(self.dimension, tfidf.shape[1] - 1)
                    self.svd = TruncatedSVD(n_components=n_components, random_state=42)
                    self.svd.fit(tfidf)
                    self.is_fitted = True
                
                tfidf = self.vectorizer.transform(texts)
                reduced = self.svd.transform(tfidf)
                
                if reduced.shape[1] < self.dimension:
                    padded = np.zeros((reduced.shape[0], self.dimension))
                    padded[:, :reduced.shape[1]] = reduced
                    reduced = padded
                
                if normalize_embeddings:
                    reduced = normalize(reduced, norm='l2')
                
                return reduced.astype(np.float32)
            
            def get_sentence_embedding_dimension(self):
                return self.dimension
        
        embedding_model = LocalEmbeddingModel()
        print(" Loaded local TF-IDF + SVD model")
    except:
        raise Exception("No embedding model available")

print(f"   Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")

# Load FAISS vector store
print("\n Loading FAISS vector store...")
vector_store_dir = '../vector_store/faiss/'

if not os.path.exists(vector_store_dir):
    vector_store_dir = 'vector_store/faiss/'

# Load FAISS index
faiss_index = faiss.read_index(f"{vector_store_dir}/index.faiss")
print(f" FAISS index loaded with {faiss_index.ntotal:,} vectors")

# Load metadata
with open(f"{vector_store_dir}/metadata.pkl", 'rb') as f:
    metadata = pickle.load(f)
print(f" Metadata loaded with {len(metadata['chunk_text'])} entries")

print(f"\n Vector Store Info:")
print(f"  Total vectors: {faiss_index.ntotal:,}")
print(f"  Embedding dimension: {faiss_index.d}")
print(f"  Metadata fields: {list(metadata.keys())}")


 Loading embedding model...


model_O1.onnx: 100%|██████████| 90.4M/90.4M [02:00<00:00, 749kB/s]
model_O2.onnx: 100%|██████████| 90.3M/90.3M [01:32<00:00, 981kB/s] 
model_O3.onnx: 100%|██████████| 90.3M/90.3M [01:25<00:00, 1.05MB/s]
model_O4.onnx: 100%|██████████| 45.2M/45.2M [00:43<00:00, 1.04MB/s]
model_qint8_arm64.onnx: 100%|██████████| 23.0M/23.0M [00:23<00:00, 995kB/s] 
model_qint8_avx512.onnx: 100%|██████████| 23.0M/23.0M [00:24<00:00, 948kB/s]
model_qint8_avx512_vnni.onnx: 100%|██████████| 23.0M/23.0M [00:22<00:00, 1.02MB/s]
model_quint8_avx2.onnx: 100%|██████████| 23.0M/23.0M [00:23<00:00, 1.00MB/s]
openvino_model.bin: 100%|██████████| 90.3M/90.3M [01:21<00:00, 1.11MB/s]
openvino_model.xml: 211kB [00:00, 183MB/s]
openvino_model_qint8_quantized.bin: 100%|██████████| 22.9M/22.9M [00:20<00:00, 1.11MB/s]
openvino_model_qint8_quantized.xml: 368kB [00:00, 23.1MB/s]
pytorch_model.bin: 100%|██████████| 90.9M/90.9M [01:27<00:00, 1.04MB/s]
sentence_bert_config.json: 100%|██████████| 53.0/53.0 [00:00<?, ?B/s]
special_

 Loaded all-MiniLM-L6-v2
   Embedding dimension: 384

 Loading FAISS vector store...
 FAISS index loaded with 36,399 vectors
 Metadata loaded with 36399 entries

 Vector Store Info:
  Total vectors: 36,399
  Embedding dimension: 384
  Metadata fields: ['chunk_text', 'complaint_id', 'product', 'issue', 'company', 'chunk_index', 'total_chunks', 'date_received']
